In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

C:\Users\manoj\PycharmProjects\LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

# Model
model = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str


In [4]:
def generate_joke(state: JokeState):
    prompt = f"Generate Hindi joke on the given topic: \n {state['topic']}"
    joke = model.invoke(prompt).content
    return {'joke': joke}


In [5]:
def generate_explanation(state: JokeState):
    prompt = f"Generate the explanation of the following joke: \n {state['joke']}"
    explanation = model.invoke(prompt).content
    return {'explanation': explanation}

In [6]:
graph = StateGraph(JokeState)

# Add Node
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

# Edges

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)


In [7]:
# Checkpointer

checkpointer = InMemorySaver()

# Compiler

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
initial_state = {
    'topic': 'pati - patni'
}

config = {'configurable':{'thread_id': 1}}

res = workflow.invoke(input=initial_state, config=config)

print(res)

{'topic': 'pati - patni', 'joke': [{'type': 'text', 'text': 'यहाँ पति-पत्नी पर आधारित कुछ मजेदार चुटकुले (Jokes) दिए गए हैं:\n\n**1. मछली और चारा**\nपत्नी: "शादी से पहले तो आप मेरे लिए बड़े-बड़े गिफ्ट लाते थे, अब क्या हुआ?"\nपति: "अरे पगली, क्या कभी किसी ने मछली पकड़ने के बाद उसे \'चारा\' खिलाया है?"\n\n**2. पड़ोस का असर**\nपत्नी: "आज मैंने खिड़की से देखा कि पड़ोस का आदमी अपनी पत्नी को \'किस\' (Kiss) कर रहा था। क्या आप ऐसा नहीं कर सकते?"\nपति: "पागल हो क्या? मैं उस औरत को जानता भी नहीं!"\n\n**3. शादी की सालगिरह**\nपत्नी: "सुनो जी, हमारी शादी के 10 साल हो गए और आज तक हमारा कोई झगड़ा नहीं हुआ। इसकी क्या वजह है?"\nपति: "वजह बहुत सिंपल है... जब तुम बोलती हो तो मैं सुनता हूँ, और जब मैं बोलता हूँ तो तुम \'सुनती\' ही नहीं हो!"\n\n**4. पंडित जी का दुख**\nपत्नी: "अजी सुनते हो, हमारी शादी कराने वाले पंडित जी का स्वर्गवास हो गया।"\nपति: "होना ही था... इतने बड़े पाप का बोझ आखिर वो कब तक सहते बेचारे!"\n\n**5. डॉक्टर की सलाह**\nपत्नी: "डॉक्टर ने मुझे एक महीने के लिए स्विट्जरलैंड जाने की सलाह दी है। 

In [11]:
workflow.get_state(config=config)

StateSnapshot(values={'topic': 'Santa - Banta', 'joke': [{'type': 'text', 'text': 'यहाँ सांता-बंता का एक मजेदार जोक है:\n\n**सांता और आईना**\n\n**सांता:** (सड़क पर गिरा हुआ एक आईना उठाता है और उसमें अपनी शक्ल देखता है)\n"ओ तेरी! ये तो मेरे बिछड़े हुए भाई की फोटो है!"\n\nसांता उस आईने को बड़े प्यार से घर ले गया और अपनी अलमारी में छुपा दिया।\n\nवह रोज़ अलमारी खोलकर उस \'फोटो\' को देखता और भावुक हो जाता।\n\nउसकी पत्नी जीतो को शक हुआ। एक दिन जब सांता घर पर नहीं था, तो जीतो ने अलमारी खोली और आईना देखा।\n\n**जीतो (गुस्से में):** "अच्छा! तो ये है वो चुड़ैल, जिसके पीछे ये पागल हुए फिर रहे हैं! शक्ल देखो इसकी... जैसे सड़ी हुई गोभी हो!"', 'extras': {'signature': 'EuoRCucRAb4+9vtEqyHJ3GNPaRFMIdnrMl0NF+79C1FAtB4c5UXJq1tbTMqnjqeZcvUfpye5gRvdw8RczBP1phS9c6eqMktTvto+nsSC/Vbb7DRJeED/3xO+lSrh77YlcZO8DiQ3m4WWbl/fbIZru+3A3VHUnlyb54ESMC81JrEw/4tqls40jfGXxxB3jMP4y4Jc2ZVXTVG5F1dVgt+EimCqZw2evDwujf6wy6qcuI+paBUjgTQClFM15VnrRYxaqEuJByuth7cqNSiOyzd04FiWk4rX8kmRwLRhBlDMTVRnZBsTP7XbV9OhM3iR7W1tKokyNC3E97Uw0rxyCU

In [16]:
list(workflow.get_state_history(config=config))

[StateSnapshot(values={'topic': 'pati - patni', 'joke': [{'type': 'text', 'text': 'यहाँ पति-पत्नी पर आधारित कुछ मजेदार चुटकुले (Jokes) दिए गए हैं:\n\n**1. मछली और चारा**\nपत्नी: "शादी से पहले तो आप मेरे लिए बड़े-बड़े गिफ्ट लाते थे, अब क्या हुआ?"\nपति: "अरे पगली, क्या कभी किसी ने मछली पकड़ने के बाद उसे \'चारा\' खिलाया है?"\n\n**2. पड़ोस का असर**\nपत्नी: "आज मैंने खिड़की से देखा कि पड़ोस का आदमी अपनी पत्नी को \'किस\' (Kiss) कर रहा था। क्या आप ऐसा नहीं कर सकते?"\nपति: "पागल हो क्या? मैं उस औरत को जानता भी नहीं!"\n\n**3. शादी की सालगिरह**\nपत्नी: "सुनो जी, हमारी शादी के 10 साल हो गए और आज तक हमारा कोई झगड़ा नहीं हुआ। इसकी क्या वजह है?"\nपति: "वजह बहुत सिंपल है... जब तुम बोलती हो तो मैं सुनता हूँ, और जब मैं बोलता हूँ तो तुम \'सुनती\' ही नहीं हो!"\n\n**4. पंडित जी का दुख**\nपत्नी: "अजी सुनते हो, हमारी शादी कराने वाले पंडित जी का स्वर्गवास हो गया।"\nपति: "होना ही था... इतने बड़े पाप का बोझ आखिर वो कब तक सहते बेचारे!"\n\n**5. डॉक्टर की सलाह**\nपत्नी: "डॉक्टर ने मुझे एक महीने के लिए स्विट्जरलैं